# 8.3 Интерактив в Plotly — код страницы

Аналитика 101 · блок «Дата аналитика» · тема 08 «Визуализация в Python» · страница 8.3.

В ноутбуке семь блоков кода со страницы. Сначала четыре ячейки раздела «Код», они идут подряд и пользуются общими переменными: выручка каналов одной функцией `px`, своя подсказка, анимация по месяцам, экспорт в HTML, JSON, PNG и SVG. Затем три нюанса, тоже подряд: вес файла с миллионом точек, вывод в заголовке вместо подсказки, график для телефона и текстовая альтернатива. Данные учебные: выручка магазина из общего набора темы `data/viz/`.

После блоков с подсказкой, анимацией и графиком для телефона добавлены ячейки `fig.show()`: на странице их нет, а в ноутбуке без них фигуру не увидеть.

**Как работать.** Выполняйте ячейки по порядку сверху вниз. Картинки PNG и SVG сохраняет пакет kaleido, ему нужен Chrome или Chromium. Если браузера на компьютере нет, его ставит команда `plotly_get_chrome`.

Вывод получен на Python 3.13, NumPy 2.5, pandas 3.0, plotly 7.1 и kaleido 1.5.

## Где лежат данные

Код страницы читает файл по пути от корня папки с материалами: `data/viz/sales_monthly.csv`. Ноутбук лежит в папке страницы, поэтому первая ячейка поднимается до папки, в которой есть `data/`, и делает её рабочей.

Туда же код сохранит свои файлы: `channels.html` (с библиотекой, почти 5 МБ), `channels_cdn.html`, `channels.json`, `channels.png`, `channels.svg`, `points.html` (миллион точек, 26 МБ), `plain.svg`, `titled.svg` и `mobile.html`. После работы их можно удалить.

In [1]:
import os
import pathlib
import platform
from importlib.metadata import version

import numpy as np
import pandas as pd
import plotly

root = pathlib.Path.cwd()
while not (root / "data" / "viz").is_dir() and root != root.parent:
    root = root.parent                      # поднимаемся, пока рядом не окажется data/viz/
assert (root / "data" / "viz" / "sales_monthly.csv").exists(), \
    "Не найдена папка data/viz/. Скачайте общий набор темы 08 и положите data/ рядом с папкой страницы"
os.chdir(root)

print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__,
      "| plotly", plotly.__version__, "| kaleido", version("kaleido"))

Python 3.13.18 | numpy 2.5.3 | pandas 3.0.5 | plotly 7.1.0 | kaleido 1.5.0


## Код: от таблицы до файла

### Пять каналов — пять трасс, а подсказку px собрал из подписей

In [2]:
import pandas as pd
import plotly.express as px

sales = pd.read_csv("data/viz/sales_monthly.csv")  # длинный формат: строка = месяц × канал × страна
top5 = ["сайт", "маркетплейс", "приложение", "поиск", "рассылка"]
by_ch = (sales[sales["channel"].isin(top5)]
         .groupby(["month", "channel"], as_index=False)["revenue"].sum()
         .assign(mln=lambda d: d["revenue"] / 1e6))
print(by_ch.shape)
print(by_ch.head(3).to_string(index=False))

fig = px.line(by_ch, x="month", y="mln", color="channel",   # столбец → роль на графике
              category_orders={"channel": top5},             # порядок в легенде
              labels={"month": "месяц", "mln": "млн ₽", "channel": "канал"},
              title="Приложение удвоило выручку после релиза в июне 2025")
fig.update_xaxes(tickformat="%m.%Y")  # 01.2025 вместо Jan 2025

app = fig.data[2]
print(type(fig).__name__, "из", len(fig.data), "трасс;", app.type, app.name, len(app.x), "точки")
print(list(fig.to_dict()), "— в браузер уходит этот словарь в виде JSON")
print(app.hovertemplate)

(120, 4)
  month     channel   revenue   mln
2024-01 маркетплейс 7471000.0 7.471
2024-01       поиск 4374000.0 4.374
2024-01  приложение 3363000.0 3.363


Figure из 5 трасс; scatter приложение 24 точки
['data', 'layout'] — в браузер уходит этот словарь в виде JSON
канал=приложение<br>месяц=%{x}<br>млн ₽=%{y}<extra></extra>


### Подсказка говорит по-русски: месяц словами, запятая и доля месяца

In [3]:
MONTHS = "январь февраль март апрель май июнь июль август сентябрь октябрь ноябрь декабрь".split()
total = sales.groupby("month")["revenue"].sum()  # выручка месяца по всем 10 каналам
by_ch["month_ru"] = [f"{MONTHS[int(m[5:]) - 1]} {m[:4]}" for m in by_ch["month"]]
by_ch["share"] = by_ch["revenue"] / by_ch["month"].map(total)

fig = px.line(by_ch, x="month", y="mln", color="channel", category_orders={"channel": top5},
              custom_data=["month_ru", "share"],  # столбцы, которые нужны только подсказке
              labels={"month": "месяц", "mln": "млн ₽", "channel": "канал"},
              title="Приложение удвоило выручку после релиза в июне 2025")
fig.update_traces(hovertemplate="<b>%{fullData.name}</b> · %{customdata[0]}<br>"
                                "%{y:.2f} млн ₽ · %{customdata[1]:.0%} выручки месяца"
                                "<extra></extra>")  # пустой extra убирает второй ярлык с именем
fig.update_layout(separators=", ")  # десятичная запятая и пробел между разрядами
fig.update_xaxes(tickformat="%m.%Y")

print(fig.data[2].hovertemplate)
print(fig.data[2].customdata[18])  # приложение, июль 2025

<b>%{fullData.name}</b> · %{customdata[0]}<br>%{y:.2f} млн ₽ · %{customdata[1]:.0%} выручки месяца<extra></extra>
['июль 2025' 0.18281999690450396]


Проверено в Chromium настоящим plotly.js из `channels.html`: подсказка у приложения в июле 2025 — «приложение · июль 2025» и строкой ниже «5,91 млн ₽ · 18% выручки месяца». У маркетплейса за июль 2025 в подсказке 5,71, а в таблице pandas из нюанса про телефон — 5,72: выручка 5 715 000 ₽ стоит ровно посередине, и JavaScript с pandas округляют её в разные стороны.

Фигура с новой подсказкой. Наведите курсор на линию приложения в июле 2025 года.

In [4]:
fig.show()

### 24 кадра, кнопки ▶ и ◼ и ползунок по месяцам

In [5]:
fig_a = px.bar(by_ch, x="channel", y="mln", color="channel",
               animation_frame="month",           # один кадр на месяц
               category_orders={"channel": top5},
               range_y=[0, 11],                   # одна шкала на все кадры
               labels={"channel": "", "mln": "млн ₽", "month": "месяц"},
               title="Выручка каналов по месяцам")
fig_a.update_layout(showlegend=False, separators=", ")

print(len(fig_a.frames), "кадра:", fig_a.frames[0].name, "…", fig_a.frames[-1].name)
print("кнопки:", [b.label for b in fig_a.layout.updatemenus[0].buttons])
print("ползунок, шаг 18:", fig_a.layout.sliders[0].steps[17].label)
print("стартовый вид = первый кадр:", list(fig_a.data[2].y) == list(fig_a.frames[0].data[2].y))
print("ось Y без range_y:", px.bar(by_ch, x="channel", y="mln", animation_frame="month").layout.yaxis.range)

24 кадра: 2024-01 … 2025-12
кнопки: ['&#9654;', '&#9724;']
ползунок, шаг 18: 2025-06
стартовый вид = первый кадр: True
ось Y без range_y: None


`&#9654;` и `&#9724;` — коды символов ▶ и ◼ в HTML. Без `range_y` диапазон оси в фигуре не записан, и plotly.js подбирает его сам: при проигрывании кнопкой ▶ в Chromium верх оси сдвинулся с 10,8 до 11,1 млн ₽, и одна высота столбика в разных кадрах значила бы разные деньги.

Анимация: кнопка ▶ под графиком проигрывает 24 месяца, ползунок выбирает месяц.

In [6]:
fig_a.show()

### Файл с библиотекой весит 4,6 МБ, сама фигура — 20 КБ

In [7]:
import os
import re

fig.write_html("channels.html")                              # include_plotlyjs=True — по умолчанию
fig.write_html("channels_cdn.html", include_plotlyjs="cdn")  # библиотеку браузер возьмёт из интернета
with open("channels.json", "w", encoding="utf-8") as f:
    f.write(fig.to_json())                                   # только фигура: данные и оформление
fig.write_image("channels.png", width=900, height=480)       # kaleido рисует через Chrome
fig.write_image("channels.svg", width=900, height=480)

for name in ["channels.html", "channels_cdn.html", "channels.json", "channels.png", "channels.svg"]:
    print(f"{name:<18} {os.path.getsize(name) / 1024:>8.1f} КБ")
print(re.search(r'src="(https://cdn[^"]+)"', open("channels_cdn.html", encoding="utf-8").read()).group(1))

channels.html        4723.7 КБ
channels_cdn.html      20.9 КБ
channels.json          19.9 КБ
channels.png           75.1 КБ
channels.svg           15.3 КБ
https://cdn.plot.ly/plotly-4.1.1.min.js


Рисунок ниже — `channels.svg` из этого блока, вставлен в страницу без правок.

## Три нюанса

На странице нюансы — отдельный ноутбук. Здесь они идут следом в той же сессии: блок про заголовок заново читает данные и строит свою фигуру `fig`.

### Миллион точек — 26 МБ данных даже без библиотеки

In [8]:
import os

import numpy as np
import plotly.express as px

rng = np.random.default_rng(83)  # учебные случайные точки: важен только их счёт
for n in [1_000, 10_000, 100_000, 1_000_000]:
    fig = px.scatter(x=rng.normal(size=n), y=rng.normal(size=n))
    fig.write_html("points.html", include_plotlyjs="cdn")  # без библиотеки: в файле только данные
    kb = os.path.getsize("points.html") / 1024
    print(f"{n:>9,} точек  {fig.data[0].type:<10} {kb:>8,.0f} КБ  {kb * 1024 / n:4.1f} байта на точку")

    1,000 точек  scatter          35 КБ  35.6 байта на точку
   10,000 точек  scattergl       274 КБ  28.0 байта на точку
  100,000 точек  scattergl     2,669 КБ  27.3 байта на точку


1,000,000 точек  scattergl    26,577 КБ  27.2 байта на точку


С `include_plotlyjs=True` к каждой строке прибавьте ещё 4,6 МБ библиотеки. Числа хранятся в JSON как двоичные массивы в base64, поэтому байтов на точку почти не зависит от того, какие это числа.

### Из 120 чисел подсказок на картинку не попало ни одного

In [9]:
import re

import pandas as pd
import plotly.express as px

sales = pd.read_csv("data/viz/sales_monthly.csv")
top5 = ["сайт", "маркетплейс", "приложение", "поиск", "рассылка"]
by_ch = (sales[sales["channel"].isin(top5)]
         .groupby(["month", "channel"], as_index=False)["revenue"].sum()
         .assign(mln=lambda d: d["revenue"] / 1e6))
ru = lambda v, d=1: f"{v:.{d}f}".replace(".", ",")
words = lambda path: re.findall(r">([^<>]+)</text>", open(path, encoding="utf-8").read())

fig = px.line(by_ch, x="month", y="mln", color="channel", title="Выручка по каналам, млн ₽")
fig.update_layout(separators=", ")
fig.write_image("plain.svg")  # то, что уйдёт в письмо или на слайд
hover = [ru(v, 2) for v in by_ch["mln"]]  # числа, которые видны только при наведении
print("чисел в подсказках:", len(hover), "| из них на картинке:", sum(h in words("plain.svg") for h in hover))

app = by_ch[by_ch["channel"] == "приложение"].set_index("month")["mln"]
fig.update_layout(title=f"Приложение: {ru(app['2025-05'])} → {ru(app['2025-07'])} млн ₽ в месяц "
                        "после релиза в июне 2025")
fig.add_annotation(x="2025-06", y=app["2025-06"], text="релиз", ay=-40)
fig.write_image("titled.svg")
print("надписи с числами:", [w for w in words("titled.svg") if "→" in w or w == "релиз"])

чисел в подсказках: 120 | из них на картинке: 0


надписи с числами: ['Приложение: 3,0 → 5,9 млн ₽ в месяц после релиза в июне 2025', 'релиз']


Надписи ищутся в SVG из kaleido: всё, что plotly.js нарисовал текстом, лежит там в тегах <text>.

### Легенда под графиком, свайп без рамки и текст вместо наведения

In [10]:
fig.update_layout(legend=dict(orientation="h", y=-0.15),  # легенда под графиком: на узком экране сбоку ей тесно
                  dragmode=False)                           # свайп прокручивает страницу, а не рисует рамку
fig.write_html("mobile.html", include_plotlyjs="cdn", config={"displayModeBar": False})

# текстовая альтернатива из тех же данных: для экранного диктора и для тех, кто не наводит
alt = (f"Линейный график выручки пяти крупнейших каналов, январь 2024 — декабрь 2025. "
       f"Приложение выросло с {ru(app['2025-05'])} млн ₽ в мае до {ru(app['2025-07'])} млн ₽ "
       f"в июле 2025 после релиза и к декабрю дошло до {ru(app['2025-12'])} млн ₽.")
print(alt)
table = by_ch.pivot(index="month", columns="channel", values="mln")[top5]
print(table.loc["2025-04":"2025-08"].round(2).to_string())

Линейный график выручки пяти крупнейших каналов, январь 2024 — декабрь 2025. Приложение выросло с 3,0 млн ₽ в мае до 5,9 млн ₽ в июле 2025 после релиза и к декабрю дошло до 8,0 млн ₽.
channel  сайт  маркетплейс  приложение  поиск  рассылка
month                                                  
2025-04  8.97         6.09        3.09   3.97      2.37
2025-05  8.77         6.02        2.98   3.72      2.24
2025-06  8.07         5.73        4.49   3.96      2.29
2025-07  8.47         5.72        5.91   4.17      2.30
2025-08  8.85         6.20        6.14   4.22      2.43


Проверено в Chromium на эмуляции Pixel 5: со стандартным `dragmode` вертикальный свайп по графику приблизил октябрь–декабрь 2024 и не прокрутил страницу, с `dragmode=False` страница прокрутилась, а оси остались прежними.

График для телефона: легенда снизу, панель инструментов скрыта, протяжка по графику не рисует рамку.

In [11]:
fig.show(config={"displayModeBar": False})